In [2]:
import cv2
import mediapipe as mp
import numpy as np
from collections import deque
import pandas as pd
import csv
import os

mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

# transform landmarks to pixel coordinates
def get_pixel_point(landmarks, index, width, height):
    return np.array([int(landmarks[index].x * width), int(landmarks[index].y * height)])

# additional feature preparation
def prepare_ml_features(lm, side, frame_width, frame_height):
    if side == "Right":
        hip, knee, ankle = 24, 26, 28
        shoulder, elbow = 12, 14
    else:
        hip, knee, ankle = 23, 25, 27
        shoulder, elbow = 11, 13

    # normalization and torso scaling
    mid_hip_x = (lm[24].x + lm[23].x) / 2
    mid_hip_y = (lm[24].y + lm[23].y) / 2
    mid_sh_x = (lm[12].x + lm[11].x) / 2
    mid_sh_y = (lm[12].y + lm[11].y) / 2

    torso_dist = np.sqrt((mid_sh_x - mid_hip_x)**2 + (mid_sh_y - mid_hip_y)**2)
    if torso_dist == 0: torso_dist = 1

    def norm_x(idx): return (lm[idx].x - mid_hip_x) / torso_dist
    def norm_y(idx): return (lm[idx].y - mid_hip_y) / torso_dist

    # feature calculation
    features = {
        'n_ankle_x': round(norm_x(ankle), 4),
        'n_ankle_y': round(norm_y(ankle), 4),
        'n_knee_x': round(norm_x(knee), 4),
        'n_knee_y': round(norm_y(knee), 4),
        'n_shoulder_y': round(norm_y(shoulder), 4),
        'n_shoulder_lean': round(norm_x(shoulder), 4),
        'n_elbow_spacing': round(abs(norm_x(elbow)), 4),
        'n_hip_diff_y': round((lm[24].y - lm[23].y) / torso_dist, 4),
        'n_shoulder_diff_y': round((lm[12].y - lm[11].y) / torso_dist, 4),
        'n_whip_gap': round(norm_x(ankle) - norm_x(knee), 4)
    }
    return features

# video features extraction
def analyze_video(video_path):
    video_capture = cv2.VideoCapture(video_path)
    fps_raw = video_capture.get(cv2.CAP_PROP_FPS)
    frames_per_second = fps_raw if fps_raw > 0 else 30.0

    # thresholds and setup
    gct_threshold = 0.015 
    strike_lockout = int(frames_per_second * 0.22) 
    history_window = 15

    right_ankle_y_history = deque(maxlen=history_window)
    left_ankle_y_history = deque(maxlen=history_window)
    cadence_history = deque(maxlen=8)
    all_step_metrics_storage = []
    
    leg_is_on_ground = {"Right": False, "Left": False}
    ground_y_level = {"Right": 0.0, "Left": 0.0}
    last_strike_frame = {"Right": 0, "Left": 0}
    previous_strike_time = None
    average_cadence = 0
    right_step_count, left_step_count = 0, 0
    current_status_event = "WAITING"
    
    peak_whip_tracker = {"Right": 0.0, "Left": 0.0}
    global_peak_foot_offset = {"Right": 0.0, "Left": 0.0}

    frame_counter = 0

    with mp_pose.Pose(min_detection_confidence=0.7, min_tracking_confidence=0.7) as pose_analyzer:
        while video_capture.isOpened():
            ret, frame = video_capture.read()
            if not ret: break
            
            frame_counter += 1
            if frame_counter % 3 == 0: continue 
                
            display_frame = frame.copy()
            overlay_layer = frame.copy()
            frame_height, frame_width, _ = frame.shape
            results = pose_analyzer.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

            if results.pose_landmarks:
                landmarks = results.pose_landmarks.landmark             

                # keypoint extraction
                right_hip = get_pixel_point(landmarks, 24, frame_width, frame_height)
                left_hip = get_pixel_point(landmarks, 23, frame_width, frame_height)
                right_shoulder = get_pixel_point(landmarks, 12, frame_width, frame_height)
                left_shoulder = get_pixel_point(landmarks, 11, frame_width, frame_height)
                right_knee = get_pixel_point(landmarks, 26, frame_width, frame_height)
                left_knee = get_pixel_point(landmarks, 25, frame_width, frame_height)
                right_ankle = get_pixel_point(landmarks, 28, frame_width, frame_height)
                left_ankle = get_pixel_point(landmarks, 27, frame_width, frame_height)
                right_elbow = get_pixel_point(landmarks, 14, frame_width, frame_height)
                left_elbow = get_pixel_point(landmarks, 13, frame_width, frame_height)
                right_wrist = get_pixel_point(landmarks, 16, frame_width, frame_height)
                left_wrist = get_pixel_point(landmarks, 15, frame_width, frame_height)

                # drawing and overlay
                torso_points = np.array([right_shoulder, left_shoulder, left_hip, right_hip], np.int32)
                cv2.fillPoly(overlay_layer, [torso_points], (0, 255, 0)) 
                cv2.addWeighted(overlay_layer, 0.15, display_frame, 0.85, 0, display_frame)
                cv2.polylines(display_frame, [torso_points], True, (255, 255, 255), 2)
                
                for h_p, k_p, a_p, color in [(right_hip, right_knee, right_ankle, (0, 255, 0)), 
                                             (left_hip, left_knee, left_ankle, (0, 255, 255))]:
                    cv2.line(display_frame, tuple(h_p), tuple(k_p), color, 3)
                    cv2.line(display_frame, tuple(k_p), tuple(a_p), color, 3)
                for s_p, e_p, w_p in [(right_shoulder, right_elbow, right_wrist), 
                                      (left_shoulder, left_elbow, left_wrist)]:
                    cv2.line(display_frame, tuple(s_p), tuple(e_p), (255, 165, 0), 3)
                    cv2.line(display_frame, tuple(e_p), tuple(w_p), (255, 165, 0), 3)

                # process step
                current_frame_pos = video_capture.get(cv2.CAP_PROP_POS_FRAMES)
                right_ankle_y_history.append(landmarks[28].y)
                left_ankle_y_history.append(landmarks[27].y)
                
                for side, history_queue, a_idx, k_idx in [("Right", right_ankle_y_history, 28, 26), 
                                                           ("Left", left_ankle_y_history, 27, 25)]:
                    curr_y = landmarks[a_idx].y
                    h_idx = 24 if side == "Right" else 23
                    
                    # foot offset angle calculation
                    v_thigh = np.array([landmarks[k_idx].x - landmarks[h_idx].x, landmarks[k_idx].y - landmarks[h_idx].y])
                    v_shin = np.array([landmarks[a_idx].x - landmarks[k_idx].x, landmarks[a_idx].y - landmarks[k_idx].y])
                    u_thigh = v_thigh / (np.linalg.norm(v_thigh) + 1e-6)
                    u_shin = v_shin / (np.linalg.norm(v_shin) + 1e-6)
                    curr_offset = np.degrees(np.arccos(np.clip(np.dot(u_thigh, u_shin), -1.0, 1.0)))
                    
                    if curr_offset > global_peak_foot_offset[side]:
                        global_peak_foot_offset[side] = curr_offset

                    if not leg_is_on_ground[side]:
                        # track maximum heel whip
                        whip = np.degrees(np.arctan2(abs(landmarks[a_idx].x - landmarks[k_idx].x), 
                                                     abs(landmarks[a_idx].y - landmarks[k_idx].y)))
                        if whip > peak_whip_tracker[side]: 
                            peak_whip_tracker[side] = whip

                        # detect strike
                        if len(history_queue) >= 10 and curr_y >= max(list(history_queue)[:-1]) and (current_frame_pos - last_strike_frame[side]) > strike_lockout:
                            leg_is_on_ground[side] = True
                            ground_y_level[side] = curr_y
                            last_strike_frame[side] = current_frame_pos
                            
                            if previous_strike_time is not None:
                                cadence_history.append((60 * frames_per_second) / (current_frame_pos - previous_strike_time))
                                average_cadence = np.mean(cadence_history)
                            previous_strike_time = current_frame_pos
                            
                            if side == "Right": right_step_count += 1
                            else: left_step_count += 1
                            current_status_event = f"{side.upper()} STRIKE"
                            
                            # store metrics
                            ml_data = prepare_ml_features(landmarks, side, frame_width, frame_height)
                            all_step_metrics_storage.append({
                                'side': side, 'start_frame': current_frame_pos, 'done': False,
                                'cadence': average_cadence, 
                                'heel_whip_val': peak_whip_tracker[side],
                                'foot_offset_val': global_peak_foot_offset[side],
                                'hip_drop_val': np.degrees(np.arctan2(landmarks[24].y - landmarks[23].y, landmarks[24].x - landmarks[23].x)),
                                'shoulder_drop_val': np.degrees(np.arctan2(landmarks[12].y - landmarks[11].y, landmarks[12].x - landmarks[11].x)),
                                **ml_data
                            })
                            peak_whip_tracker[side] = 0.0

                    elif leg_is_on_ground[side]:
                        # detect push-off
                        if (ground_y_level[side] - curr_y) > gct_threshold:
                            for entry in reversed(all_step_metrics_storage):
                                if entry['side'] == side and not entry['done']:
                                    entry['gct'] = ((current_frame_pos - entry['start_frame']) / frames_per_second) * 1000
                                    entry['done'] = True
                                    leg_is_on_ground[side] = False
                                    current_status_event = f"{side.upper()} PUSH-OFF"
                                    break

                # display visualizations
                cv2.rectangle(display_frame, (0, 0), (280, 100), (20, 20, 20), -1)
                cv2.putText(display_frame, f"STEPS: {right_step_count + left_step_count}", (15, 35), 1, 1.8, (255, 255, 255), 2)
                cv2.putText(display_frame, f"CADENCE: {int(average_cadence)}", (15, 65), 1, 1.2, (0, 255, 0), 2)
                cv2.putText(display_frame, f"STATUS: {current_status_event}", (15, 95), 1, 1.0, (0, 255, 255), 1)

                for side_name, pos, grounded in [("LEFT", (10, frame_height-20), leg_is_on_ground["Left"]), 
                                                  ("RIGHT", (frame_width-135, frame_height-20), leg_is_on_ground["Right"])]:
                    status_color = (0, 255, 0) if grounded else (0, 0, 255)
                    cv2.rectangle(display_frame, (pos[0]-10, pos[1]-60), (pos[0]+125, pos[1]+10), (0,0,0), -1)
                    cv2.putText(display_frame, side_name, (pos[0], pos[1]-40), 1, 1.2, status_color, 2)
                    cv2.putText(display_frame, "CONTACT" if grounded else "FLIGHT", (pos[0], pos[1]-20), 1, 0.9, (255,255,255), 1)

                cv2.imshow('Running Analysis', display_frame)
                if cv2.waitKey(1) & 0xFF == ord('q'): break

    video_capture.release()
    cv2.destroyAllWindows()
    return all_step_metrics_storage

In [3]:
# baseline score intervals
def get_metric_score(metric, val):
    val = abs(val)
    
    if metric == "cadence":
        if val >= 175: return 3
        if val >= 165: return 2
        if val >= 150: return 1
        return 0
    if metric == "heel_whip":
        if val < 7.0: return 3
        if 7.0 <= val <= 10.0: return 2
        if 10.0 <= val <= 15.0: return 1
        return 0
    if metric == "foot_offset":
        if val < 6.0: return 3
        if 6.0 <= val <= 9.0: return 2
        if 9.0 <= val <= 13.0: return 1
        return 0
    if metric == "hip_drop":
        if val < 5.0: return 3
        if 5.0 <= val <= 7.0: return 2
        if 7.0 <= val <= 10.0: return 1
        return 0
    if metric == "shoulder_drop":
        if val < 7.0: return 3
        if 7.0 <= val <= 10.0: return 2
        if 10.0 <= val <= 15.0: return 1
        return 0
    return 0

In [4]:
# csv storing
fieldnames = [
    'rep_number', 'frame_index', 'file_name', 'side',
    'cadence_val', 'heel_whip_val (deg)', 'foot_offset_val (deg)', 
    'hip_drop_val (deg)', 'shoulder_drop_val (deg)',
    'n_ankle_x', 'n_ankle_y', 'n_knee_x', 'n_knee_y', 'n_shoulder_y',
    'n_shoulder_lean', 'n_elbow_spacing', 'n_hip_diff_y', 
    'n_shoulder_diff_y', 'n_whip_gap',
    'cadence_score', 'heel_whip_score', 'foot_offset_score', 
    'hip_drop_score', 'shoulder_drop_score'
]

def save_steps_to_csv(steps_list, video_path, output_file='dataset_back_view.csv'):
    file_exists = os.path.isfile(output_file)
    
    line_count = 0
    if file_exists:
        with open(output_file, 'r') as f:
            line_count = sum(1 for _ in f) - 1
    # file set up
    with open(output_file, 'a', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()

        processed_count = 0
        for i, m in enumerate(steps_list, start=max(0, line_count) + 1):
            if not m.get('done'): continue
            
            # row data mapping
            row = {
                'rep_number': i,
                'frame_index': int(m['start_frame']),
                'file_name': video_path,
                'side': m['side'],
                
                # raw values
                'cadence_val': round(m.get('cadence', 0), 1),
                'heel_whip_val (deg)': round(m.get('heel_whip_val', 0), 1),
                'foot_offset_val (deg)': round(m.get('foot_offset_val', 0), 1),
                'hip_drop_val (deg)': round(m.get('hip_drop_val', 0), 1),
                'shoulder_drop_val (deg)': round(m.get('shoulder_drop_val', 0), 1),
                
                # ml feature mapping
                'n_ankle_x': m.get('n_ankle_x'),
                'n_ankle_y': m.get('n_ankle_y'),
                'n_knee_x': m.get('n_knee_x'),
                'n_knee_y': m.get('n_knee_y'),
                'n_shoulder_y': m.get('n_shoulder_y'),
                'n_shoulder_lean': m.get('n_shoulder_lean'),
                'n_elbow_spacing': m.get('n_elbow_spacing'),
                'n_hip_diff_y': m.get('n_hip_diff_y'),
                'n_shoulder_diff_y': m.get('n_shoulder_diff_y'),
                'n_whip_gap': m.get('n_whip_gap'),
                
                # score calculation - labels
                'cadence_score': get_metric_score("cadence", m.get('cadence', 0)),
                'heel_whip_score': get_metric_score("heel_whip", m.get('heel_whip_val', 0)),
                'foot_offset_score': get_metric_score("foot_offset", m.get('foot_offset_val', 0)),
                'hip_drop_score': get_metric_score("hip_drop", m.get('hip_drop_val', 0)),
                'shoulder_drop_score': get_metric_score("shoulder_drop", m.get('shoulder_drop_val', 0))
            }
            
            writer.writerow(row)
            processed_count += 1
            
    print(f"data export: successfully added {processed_count} steps to {output_file}")

In [5]:
# back view dataset creation
paths = []

for i in range(1, 21):
    paths.append(f"./back_view_dataset_videos/{i}.mov")

for video_path in paths:
    # data processing
    captured_data = analyze_video(video_path)

    # dataset saving and scoring
    if captured_data:
        save_steps_to_csv(captured_data, video_path, output_file='dataset_backview.csv')

print("Dataset updated.")

I0000 00:00:1769709339.449898  790549 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 88.1), renderer: Apple M3
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1769709339.516939  792139 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1769709339.526901  792139 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1769709339.539527  792144 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


data export: successfully added 106 steps to dataset_backview.csv


I0000 00:00:1769709388.227000  790549 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 88.1), renderer: Apple M3
W0000 00:00:1769709388.299103  793415 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1769709388.310323  793417 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


data export: successfully added 166 steps to dataset_backview.csv


I0000 00:00:1769709450.780733  790549 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 88.1), renderer: Apple M3
W0000 00:00:1769709450.848319  794596 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1769709450.857311  794596 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


...

data export: successfully added 82 steps to dataset_backview.csv


I0000 00:00:1769709894.107638  790549 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 88.1), renderer: Apple M3
W0000 00:00:1769709894.172912  805396 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1769709894.182912  805399 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


data export: successfully added 47 steps to dataset_backview.csv


I0000 00:00:1769709933.780550  790549 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 88.1), renderer: Apple M3
W0000 00:00:1769709933.851803  806170 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1769709933.861756  806170 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


data export: successfully added 83 steps to dataset_backview.csv
Dataset updated.


In [6]:
# model training and evaluating
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# loading data
df = pd.read_csv('dataset_backview.csv')

# feature and target definition
X_features = [
    'cadence_val', 'heel_whip_val (deg)', 'foot_offset_val (deg)', 
    'hip_drop_val (deg)', 'shoulder_drop_val (deg)',
    'n_ankle_x', 'n_ankle_y', 'n_knee_x', 'n_knee_y', 'n_shoulder_y',
    'n_shoulder_lean', 'n_elbow_spacing', 'n_hip_diff_y', 
    'n_shoulder_diff_y', 'n_whip_gap'
]

target_scores = [
    'cadence_score', 'heel_whip_score', 'foot_offset_score', 
    'hip_drop_score', 'shoulder_drop_score'
]

# data cleaning
df_clean = df[df['cadence_val'] > 0].dropna()
X = df_clean[X_features]
Y = df_clean[target_scores]

# train test split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# model training and evaluation
print(f"═" * 40)
print(f"{'metric':<22} | {'accuracy':<10}")
print(f"─" * 40)

models = {}
for target in target_scores:
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train, Y_train[target])
    
    y_pred = clf.predict(X_test)
    acc = accuracy_score(Y_test[target], y_pred)
    
    models[target] = clf
    print(f"{target:<22} | {acc:.2%}")

print(f"═" * 45)

# sample prediction
sample_idx = 0
real_values = Y_test.iloc[sample_idx].values
predicted_values = [models[t].predict(X_test.iloc[[sample_idx]])[0] for t in target_scores]

print("\none step check:")
print(f"real scores: {real_values}")
print(f"predicted scores: {predicted_values}")

════════════════════════════════════════
metric                 | accuracy  
────────────────────────────────────────
cadence_score          | 100.00%
heel_whip_score        | 90.91%
foot_offset_score      | 90.91%
hip_drop_score         | 90.91%
shoulder_drop_score    | 68.18%
═════════════════════════════════════════════

one step check:
real scores: [0 3 0 0 0]
predicted scores: [0, 3, 0, 0, 3]


In [7]:
# model testing
def test_on_back_video(video_path, trained_models, features_list):
    print(f"analyzing video: {video_path}")
    
    # video analysis and data extraction
    raw_steps_data = analyze_video(video_path) 
    if not raw_steps_data:
        print("no steps detected.")
        return

    # dataframe preparation
    test_df = pd.DataFrame(raw_steps_data)
    test_df = test_df[test_df['done'] == True].copy()

    column_mapping = {
        'cadence': 'cadence_val',
        'heel_whip_val': 'heel_whip_val (deg)',
        'foot_offset_val': 'foot_offset_val (deg)',
        'hip_drop_val': 'hip_drop_val (deg)',
        'shoulder_drop_val': 'shoulder_drop_val (deg)'
    }
    model_df = test_df.rename(columns=column_mapping)

    # target mapping for math score comparison
    target_to_math = {
        'cadence_score': 'cadence',
        'heel_whip_score': 'heel_whip_val',
        'foot_offset_score': 'foot_offset_val',
        'hip_drop_score': 'hip_drop_val',
        'shoulder_drop_score': 'shoulder_drop_val'
    }

    print(f"\n{'metric':<22} | {'prediction':<15} | {'calculation'}")
    print("-" * 55)
    
    # evaluation loop
    for target, model in trained_models.items():
        # prediction using
        X_input = model_df[features_list]
        ai_preds = model.predict(X_input)
        avg_ai_score = int(round(np.mean(ai_preds)))
        
        # compare with math logic
        math_key = target_to_math.get(target)
        if math_key in test_df.columns:
            avg_val = test_df[math_key].mean()
            math_score = get_metric_score(math_key, avg_val)
        else:
            math_score = "N/A"
            
        print(f"{target:<22} | {avg_ai_score:<15} | {math_score}")

# execution
test_on_back_video('./Videos_back_view/back_view_bad.mov', models, X_features)

analyzing video: ./Videos_back_view/back_view_bad.mov


I0000 00:00:1769519673.250501   31331 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 88.1), renderer: Apple M3
W0000 00:00:1769519673.327761   35269 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1769519673.337277   35275 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.



metric                 | prediction      | calculation
-------------------------------------------------------
cadence_score          | 0               | 2
heel_whip_score        | 1               | 0
foot_offset_score      | 0               | 0
hip_drop_score         | 3               | 0
shoulder_drop_score    | 2               | 0
